<a href="https://colab.research.google.com/github/kirusha20220932-cm/CRM-CW1_20220932/blob/main/CRM_CM_BI_CW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install yfinance

import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
import datetime

# library versions used
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("yfinance:", yf.__version__)

pandas: 2.2.3
numpy: 2.4.6
yfinance: 0.2.66


In [2]:
# Part 1: get the S&P 500 list from Wikipedia
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
table = pd.read_html(url, storage_options={"User-Agent": "Mozilla/5.0"})
sp500_df = table[0]

retrieval_date = datetime.date.today()
print("Retrieval date:", retrieval_date)

# yahoo uses '-' instead of '.' (BRK.B -> BRK-B)
sp500_df['Ticker'] = sp500_df['Symbol'].str.replace('.', '-', regex=False)
tickers = sp500_df['Ticker'].tolist()
print("Number of tickers:", len(tickers))

# save a copy of the list
sp500_df.to_csv('sp500_tickers.csv', index=False)
sp500_df[['Symbol', 'Security', 'GICS Sector']].head()

Retrieval date: 2026-11-21
Number of tickers: 503


,Symbol,Security,GICS Sector
0,MMM,3M,Industrials
1,AOS,A. O. Smith,Industrials
2,ABT,Abbott Laboratories,Health Care
3,ABBV,AbbVie,Health Care
4,ACN,Accenture,Information Technology


In [3]:
# Part 2: download daily data for all shares and the S&P 500 index
start = '2021-01-01'
end = '2025-01-01'     # end date not included, so last day is 31 Dec 2024

data = yf.download(tickers + ['^GSPC'], start=start, end=end,
                   auto_adjust=False, actions=True)

df_close = data['Close']
df_adj = data['Adj Close']
df_volume = data['Volume']
df_splits = data['Stock Splits']
df_dividends = data['Dividends']

print("Shape (days, tickers):", df_adj.shape)
print("First date:", df_adj.index.min().date(), "| Last date:", df_adj.index.max().date())

# tickers that didn't download anything
failed = [t for t in df_adj.columns if df_adj[t].isna().all()]
print("Tickers with no data:", failed)

# save the raw data so I don't have to download again
df_close.to_csv('close.csv')
df_adj.to_csv('adj_close.csv')
df_volume.to_csv('volume.csv')
df_splits.to_csv('splits.csv')
df_dividends.to_csv('dividends.csv')

[*****                 10%                       ]  52 of 504 completed/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
[*********************100%***********************]  504 of 504 completed
ERROR:yfinance:
5 Failed downloads:
ERROR:yfinance:['VYLR', 'SNDK', 'FDXF', 'Q', 'HONA']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')


Shape (days, tickers): (1005, 504)
First date: 2021-01-04 | Last date: 2024-12-31
Tickers with no data: ['FDXF', 'HONA', 'Q', 'SNDK', 'VYLR']


In [4]:
# Part 3: data quality checks
shares = [t for t in tickers if t not in failed]

# duplicate dates
print("Duplicate dates:", df_adj.index.duplicated().sum())

# missing values in each share
missing_adj = df_adj[shares].isna().sum()
missing_close = df_close[shares].isna().sum()
missing_vol = df_volume[shares].isna().sum()
print("Shares with missing Adj Close:", (missing_adj > 0).sum())

# unequal histories - when does each share's data start?
first_date = pd.Series({t: df_adj[t].first_valid_index() for t in shares})
late_start = first_date[first_date > df_adj.index.min()].sort_values()
print("Shares that start after 4 Jan 2021:", len(late_start))
print(late_start)

# zero or negative prices
bad_price = ((df_adj[shares] <= 0) | (df_close[shares] <= 0)).sum()
print("Shares with zero/negative prices:", (bad_price > 0).sum())

# volume problems
neg_vol = (df_volume[shares] < 0).sum()
zero_vol = (df_volume[shares] == 0).sum()
print("Shares with negative volume:", (neg_vol > 0).sum())
print("Shares with zero-volume days:", (zero_vol > 0).sum())

# which shares have zero-volume days, and when
zero_shares = zero_vol[zero_vol > 0].index.tolist()
for t in zero_shares:
    print(t, "->", df_volume.index[df_volume[t] == 0].date.tolist())

# corporate actions - splits and dividends
split_count = (df_splits[shares] > 0).sum()
div_count = (df_dividends[shares] > 0).sum()
print("Shares with a stock split:", (split_count > 0).sum())
print(split_count[split_count > 0])
print("Shares that paid dividends:", (div_count > 0).sum())

# benchmark check
print("^GSPC missing days:", df_adj['^GSPC'].isna().sum())

Duplicate dates: 0
Shares with missing Adj Close: 10
Shares that start after 4 Jan 2021: 10
EXE    2021-02-10
COIN   2021-04-14
APP    2021-04-15
HOOD   2021-07-29
CEG    2022-01-19
GEHC   2022-12-15
VLTO   2023-10-04
RDDT   2024-03-21
SOLV   2024-03-26
GEV    2024-03-27
dtype: datetime64[ns]
Shares with zero/negative prices: 0
Shares with negative volume: 0
Shares with zero-volume days: 2
FERG -> [datetime.date(2021, 1, 7), datetime.date(2021, 1, 8), datetime.date(2021, 1, 12), datetime.date(2021, 1, 13), datetime.date(2021, 1, 14), datetime.date(2021, 1, 15), datetime.date(2021, 1, 20), datetime.date(2021, 1, 21), datetime.date(2021, 1, 22), datetime.date(2021, 1, 25), datetime.date(2021, 1, 26), datetime.date(2021, 1, 28), datetime.date(2021, 1, 29), datetime.date(2021, 2, 1), datetime.date(2021, 2, 2), datetime.date(2021, 2, 3), datetime.date(2021, 2, 9), datetime.date(2021, 2, 10), datetime.date(2021, 2, 11), datetime.date(2021, 2, 12), datetime.date(2021, 2, 16), datetime.date(20

In [5]:
# Part 4: inclusion / exclusion log
log = []
for t in tickers:
    if t in failed:
        reason = 'No price data for 2021-2024'
    elif t in late_start.index:
        reason = 'Started trading after 4 Jan 2021 (' + str(late_start[t].date()) + ')'
    elif missing_adj[t] > 0 or missing_close[t] > 0 or missing_vol[t] > 0:
        reason = 'Gaps in the data'
    elif bad_price[t] > 0:
        reason = 'Zero or negative price'
    elif neg_vol[t] > 0:
        reason = 'Negative volume'
    elif zero_vol[t] > 0:
        reason = 'Zero-volume days (' + str(zero_vol[t]) + ') - not fully trading on the US market'
    else:
        reason = 'Included'
    status = 'Included' if reason == 'Included' else 'Excluded'
    log.append({'Ticker': t, 'Status': status, 'Reason': reason})

log_df = pd.DataFrame(log)
log_df.to_csv('inclusion_exclusion_log.csv', index=False)

print(log_df['Status'].value_counts())
print(log_df[log_df['Status'] == 'Excluded'].to_string(index=False))

valid_shares = log_df.loc[log_df['Status'] == 'Included', 'Ticker'].tolist()
print("Valid shares:", len(valid_shares))

Status
Included    486
Excluded     17
Name: count, dtype: int64
Ticker   Status                                                      Reason
   APP Excluded               Started trading after 4 Jan 2021 (2021-04-15)
  COIN Excluded               Started trading after 4 Jan 2021 (2021-04-14)
   CEG Excluded               Started trading after 4 Jan 2021 (2022-01-19)
   EXE Excluded               Started trading after 4 Jan 2021 (2021-02-10)
  FDXF Excluded                                 No price data for 2021-2024
  FERG Excluded  Zero-volume days (27) - not fully trading on the US market
  GEHC Excluded               Started trading after 4 Jan 2021 (2022-12-15)
   GEV Excluded               Started trading after 4 Jan 2021 (2024-03-27)
  HONA Excluded                                 No price data for 2021-2024
     Q Excluded                                 No price data for 2021-2024
  RDDT Excluded               Started trading after 4 Jan 2021 (2024-03-21)
  HOOD Excluded        

In [6]:
# Part 5: weekly prices and weekly returns
def calculate_weekly_returns(prices):
    """Weekly returns from daily prices (dates as rows, tickers as columns)."""
    weekly_prices = prices.resample("W-FRI").last()   # last price of each Friday-ending week

    # drop the first / last week if the data doesn't cover the whole week
    first_day = prices.index.min()
    last_day = prices.index.max()
    if first_day > weekly_prices.index[0] - pd.Timedelta(days=4):
        weekly_prices = weekly_prices.iloc[1:]
    if last_day < weekly_prices.index[-1]:
        weekly_prices = weekly_prices.iloc[:-1]

    weekly_returns = weekly_prices.pct_change(fill_method=None)
    return weekly_prices, weekly_returns.iloc[1:]      # first week has no previous week

weekly_prices, weekly_returns = calculate_weekly_returns(df_adj[valid_shares + ['^GSPC']])

print("Weekly prices from", weekly_prices.index.min().date(), "to", weekly_prices.index.max().date())
print("Number of weekly returns:", len(weekly_returns))
print("Missing weekly returns:", weekly_returns.isna().sum().sum())
weekly_returns[['AAPL', 'MSFT', '^GSPC']].head()

Weekly prices from 2021-01-08 to 2024-12-27
Number of weekly returns: 207
Missing weekly returns: 0


Ticker,AAPL,MSFT,^GSPC
Date,,,
2021-01-15,-0.037183,-0.031737,-0.014754
2021-01-22,0.093833,0.062544,0.019431
2021-01-29,-0.051125,0.026599,-0.033120
2021-02-05,0.037923,0.044146,0.046467
2021-02-12,-0.010164,0.011520,0.012349


In [7]:
# Part 6: mean weekly return and weekly beta
mean_return = weekly_returns[valid_shares].mean()
return_weeks = weekly_returns[valid_shares].count()

def calculate_weekly_beta(weekly_returns, market_returns, min_weeks=52):
    """Beta = cov(share, market) / var(market) on matching weeks."""
    results = []
    for share in weekly_returns.columns:
        paired = pd.concat([weekly_returns[share], market_returns], axis=1)
        paired = paired.replace([np.inf, -np.inf], np.nan).dropna()
        paired.columns = ['share', 'market']

        if len(paired) >= min_weeks and paired['market'].var() > 0:
            beta = paired['share'].cov(paired['market']) / paired['market'].var()
        else:
            beta = np.nan
        results.append({'Share': share, 'Beta': beta, 'Aligned_weeks': len(paired)})
    return pd.DataFrame(results).set_index('Share')

beta_df = calculate_weekly_beta(weekly_returns[valid_shares], weekly_returns['^GSPC'])

# put both features in one table
risk_df = pd.DataFrame({'Mean_weekly_return': mean_return,
                        'Return_weeks': return_weeks}).join(beta_df)
risk_df.to_csv('risk_return_features.csv')

print("Shares:", len(risk_df))
print("Aligned weeks - min:", risk_df['Aligned_weeks'].min(), "max:", risk_df['Aligned_weeks'].max())
print(risk_df[['Mean_weekly_return', 'Beta']].describe().round(4))

Shares: 486
Aligned weeks - min: 207 max: 207
       Mean_weekly_return      Beta
count            486.0000  486.0000
mean               0.0029    0.9737
std                0.0027    0.4073
min               -0.0045    0.0555
25%                0.0012    0.6666
50%                0.0026    0.9506
75%                0.0042    1.1795
max                0.0178    3.7786


In [8]:
# Part 7: check one share by hand (AAPL)
p0 = weekly_prices.loc[weekly_prices.index[0], 'AAPL']
p1 = weekly_prices.loc[weekly_prices.index[1], 'AAPL']
print("AAPL price", weekly_prices.index[0].date(), "=", round(p0, 2))
print("AAPL price", weekly_prices.index[1].date(), "=", round(p1, 2))
print("Return by hand:", round(p1 / p0 - 1, 6), "| from code:", round(weekly_returns['AAPL'].iloc[0], 6))

# beta check: slope of the line between AAPL and market returns
slope = np.polyfit(weekly_returns['^GSPC'], weekly_returns['AAPL'], 1)[0]
print("AAPL beta from code:", round(risk_df.loc['AAPL', 'Beta'], 4), "| slope check:", round(slope, 4))

AAPL price 2021-01-08 = 128.1
AAPL price 2021-01-15 = 123.34
Return by hand: -0.037183 | from code: -0.037183
AAPL beta from code: 1.1748 | slope check: 1.1748


In [9]:
# Part 8: observation date and high-volume days
# observation date = last date that every valid share has data for
last_dates = pd.Series({t: df_volume[t].last_valid_index() for t in valid_shares})
obs_date = last_dates.min()
print("Observation date:", obs_date.date())

# high-volume day: volume > 2 x average of the previous 20 days (today not included)
vol = df_volume[valid_shares]
baseline = vol.shift(1).rolling(20).mean()
high_volume = vol > 2 * baseline

print("Days with a full 20-day baseline start on:", baseline.dropna().index[0].date())
print("Total high-volume days (all shares, whole period):", int(high_volume.sum().sum()))

Observation date: 2024-12-31
Days with a full 20-day baseline start on: 2021-02-02
Total high-volume days (all shares, whole period): 18280


In [10]:
# Part 9: Recency, Frequency, Monetary
window = vol.index[vol.index <= obs_date][-60:]      # last 60 trading days
print("60-day window:", window[0].date(), "to", window[-1].date(), "| sessions:", len(window))

results = []
for t in valid_shares:
    events = high_volume.index[high_volume[t] & (high_volume.index <= obs_date)]

    if len(events) > 0:
        recency = (obs_date - events[-1]).days        # calendar days
    else:
        recency = np.nan                               # no event -> missing, not 0

    frequency = int(high_volume.loc[window, t].sum())
    monetary = (df_close.loc[window, t] * df_volume.loc[window, t]).sum()

    results.append({'Share': t, 'Recency': recency, 'Frequency': frequency,
                    'Monetary': monetary, 'Last_event': events[-1].date() if len(events) > 0 else None})

rfm_df = pd.DataFrame(results).set_index('Share')
rfm_df['Recency'] = rfm_df['Recency'].astype('Int64')
rfm_df.to_csv('rfm_features.csv')

print("Shares:", len(rfm_df))
print("Recency missing (no event):", rfm_df['Recency'].isna().sum())
print("Recency = 0 (event on observation date):", (rfm_df['Recency'] == 0).sum())
print(rfm_df[['Recency', 'Frequency', 'Monetary']].describe().round(1))

60-day window: 2024-10-07 to 2024-12-31 | sessions: 60
Shares: 486
Recency missing (no event): 0
Recency = 0 (event on observation date): 0
       Recency  Frequency      Monetary
count    486.0      486.0  4.860000e+02
mean      20.4        3.0  3.940419e+10
std       21.3        1.3  1.291388e+11
min        4.0        0.0  1.297483e+09
25%       11.0        2.0  1.019517e+10
50%       11.0        3.0  1.668382e+10
75%       14.8        4.0  3.227689e+10
max      222.0        8.0  1.847697e+12


In [11]:
# Part 10: check the high-volume rule on AAPL's latest event
t = 'AAPL'
last_event = rfm_df.loc[t, 'Last_event']
pos = vol.index.get_loc(pd.Timestamp(last_event))

check = pd.DataFrame({'Volume': vol[t], '20d_avg_before': baseline[t],
                      'Threshold_2x': 2 * baseline[t], 'High_volume': high_volume[t]})
print(check.iloc[pos-3:pos+2])

# baseline by hand = mean of the 20 days BEFORE the event day
by_hand = vol[t].iloc[pos-20:pos].mean()
print("20-day average by hand:", round(by_hand), "| from code:", round(baseline[t].iloc[pos]))
print("AAPL R, F, M:", rfm_df.loc[t, ['Recency', 'Frequency', 'Monetary']].tolist())

               Volume  20d_avg_before  Threshold_2x  High_volume
Date                                                            
2024-12-17   51356400      42354720.0    84709440.0        False
2024-12-18   56774100      42690855.0    85381710.0        False
2024-12-19   60882300      43718970.0    87437940.0        False
2024-12-20  147495300      45004605.0    90009210.0         True
2024-12-23   40858800      50273955.0   100547910.0        False
20-day average by hand: 45004605 | from code: 45004605
AAPL R, F, M: [np.int64(11), np.int64(2), np.float64(623303377706.5842)]


In [12]:
# Part 11: worked RFM example for one share
t = 'AAPL'
win = pd.DataFrame({'Close': df_close.loc[window, t],
                    'Volume': df_volume.loc[window, t],
                    'Threshold_2x': 2 * baseline.loc[window, t],
                    'High_volume': high_volume.loc[window, t]})
win['Traded_value'] = win['Close'] * win['Volume']

# F: high-volume days inside the 60-day window
print("High-volume days in the 60-day window:")
print(win[win['High_volume']])
print("F =", int(win['High_volume'].sum()))

# R: calendar days from the last event to the observation date
last_event = pd.Timestamp(rfm_df.loc[t, 'Last_event'])
print("Last event:", last_event.date(), "| Observation date:", obs_date.date(),
      "| R =", (obs_date - last_event).days, "days")

# M: add up Close x Volume over the 60 days
print("First 3 days of the window:")
print(win[['Close', 'Volume', 'Traded_value']].head(3))
print("M =", round(win['Traded_value'].sum() / 1e9, 2), "billion USD")

High-volume days in the 60-day window:
                 Close     Volume  Threshold_2x  High_volume  Traded_value
Date                                                                      
2024-11-25  232.869995   90152800    87641480.0         True  2.099388e+10
2024-12-20  254.490005  147495300    90009210.0         True  3.753608e+10
F = 2
Last event: 2024-12-20 | Observation date: 2024-12-31 | R = 11 days
First 3 days of the window:
                 Close    Volume  Traded_value
Date                                          
2024-10-07  221.690002  39505400  8.757952e+09
2024-10-08  225.770004  31855700  7.192062e+09
2024-10-09  229.539993  33591100  7.710501e+09
M = 623.3 billion USD


In [13]:
# Part 12: feature table - one row per share
feature_df = risk_df[['Mean_weekly_return', 'Beta', 'Aligned_weeks']].join(
             rfm_df[['Recency', 'Frequency', 'Monetary']])
feature_df['Sector'] = sp500_df.set_index('Ticker')['GICS Sector']
feature_df.to_csv('feature_table.csv')

print("Feature table:", feature_df.shape)
print("Missing values:")
print(feature_df.isna().sum())
print(feature_df[['Mean_weekly_return', 'Beta', 'Recency', 'Frequency', 'Monetary']].describe().round(4))

# which shares are at the extremes
for col in ['Mean_weekly_return', 'Beta', 'Recency', 'Frequency', 'Monetary']:
    print(col, "-> highest:", feature_df[col].idxmax(), "| lowest:", feature_df[col].idxmin())

Feature table: (486, 7)
Missing values:
Mean_weekly_return    0
Beta                  0
Aligned_weeks         0
Recency               0
Frequency             0
Monetary              0
Sector                0
dtype: int64
       Mean_weekly_return      Beta  Recency  Frequency      Monetary
count            486.0000  486.0000    486.0   486.0000  4.860000e+02
mean               0.0029    0.9737  20.4074     2.9897  3.940419e+10
std                0.0027    0.4073  21.3424     1.3308  1.291388e+11
min               -0.0045    0.0555      4.0     0.0000  1.297483e+09
25%                0.0012    0.6666     11.0     2.0000  1.019517e+10
50%                0.0026    0.9506     11.0     3.0000  1.668382e+10
75%                0.0042    1.1795    14.75     4.0000  3.227689e+10
max                0.0178    3.7786    222.0     8.0000  1.847697e+12
Mean_weekly_return -> highest: SMCI | lowest: EL
Beta -> highest: CVNA | lowest: GIS
Recency -> highest: NVDA | lowest: AJG
Frequency -> highest: CI 